In [4]:
%pip install --upgrade "google-meridian[and-cuda]"
# installs meridian 

  Using cached google_meridian-1.5.3-py3-none-any.whl.metadata (10 kB)
  Using cached arviz-0.19.0-py3-none-any.whl.metadata (8.9 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached bidict-0.23.1-py3-none-any.whl.metadata (8.7 kB)
  Using cached immutabledict-4.3.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached natsort-7.1.1-py3-none-any.whl.metadata (22 kB)
  Using cached numpy-2.3.5-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached statsmodels-0.14.6-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.5 kB)
INFO: pip is looking at multiple versions of google-meridian[and-cuda] to determine which version is compatible with other requirements. This could take a while.
  Using cached google_meridian-1.5.2-py3-none-any.whl.metadata (10 kB)
  Using cached arviz-1.0.0-py3-none-any.whl.metadata (7.5 kB)
  Usin

In [ ]:
import meridian
import IPython

import pandas as pd

from meridian import constants

from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec

from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer

from meridian.analysis.review import reviewer

import tensorflow as tf
import tensorflow_probability as tfp

print(meridian.__version__)

ModuleNotFoundError: No module named 'meridian'

In [0]:
df = pd.read_csv(
    # Optionally, use `f"${meridian_root}/"` to load data from the mounted storage.
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv"
)

# Print few rows 
print(f"Sample:\n{df.head()}")

# Print feature names 
print(f"Features:\n{df.columns}")


In [0]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)


builder = (
    builder.with_kpi(df)
    .with_revenue_per_kpi(df)
    .with_population(df)
    .with_controls(
        df, control_cols=["sentiment_score_control", "competitor_sales_control"]
    )
)

channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
builder = builder.with_media(
    df,
    media_cols=[f"{channel}_impression" for channel in channels],
    media_spend_cols=[f"{channel}_spend" for channel in channels],
    media_channels=channels,
)

builder = builder.with_non_media_treatments(
    df, non_media_treatment_cols=['Promo']
).with_organic_media(
    df,
    organic_media_cols=['Organic_channel0_impression'],
    organic_media_channels=['Organic_channel0'],
)


data = builder.build()

print(f"Data:\n{data}")

In [0]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, enable_aks=False)

mmm = model.Meridian(input_data=data, model_spec=model_spec)

In [0]:
%%time
mmm.sample_prior(50)
mmm.sample_posterior(
    n_chains=1, n_adapt=100, n_burnin=50, n_keep=100, seed=0
)
     

In [0]:
meridian_root = "/tmp/"  # Databricks recommended path for file outputs

In [0]:
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

In [0]:

model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

In [0]:

model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

In [0]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [0]:
filepath = meridian_root
start_date = '2021-01-25'
end_date = '2024-01-15'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

In [0]:
IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

In [0]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

In [0]:
budget_optimizer

In [0]:
filepath = meridian_root
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [0]:
IPython.display.HTML(filename=f'{meridian_root}/optimization_output.html')